# EBM Silver Fine-Tuning v2 (LOCAL GPU)

**This is the LOCAL version of the silver fine-tuning v2 notebook**, intended to run on a Windows workstation with an NVIDIA GPU (tested on RTX 4050, 6 GB VRAM, CUDA 12.4).
The Colab version is in `Graph_EBM_v3_Colab.ipynb`.

**Objective:** Fine-tune a gold-pretrained EBM model on silver scenarios using Contrastive Divergence + Preference loss with LP oracle evaluation.

**Silver Fine-Tuning v2:**
- Ranking-driven fine-tuning from the best stage-1 silver checkpoint
- Uses `LPWorkerTwoStage` to generate preference pairs based on LP cost + slack
- Multiple candidates per scenario with informative pair filtering
- Validation on ranking metrics (Spearman correlation, preference accuracy)

## Prerequisites (one-time setup)

### System
- Windows 10/11 with an NVIDIA GPU and recent driver (>= 552.x for CUDA 12.4 runtime).
- Python 3.10 or 3.11 (64-bit). Avoid 3.12 (no torch-scatter wheels).
- (Optional) Microsoft Visual C++ Build Tools 2022 — only if a torch-scatter wheel falls back to source build.

### Python environment (PowerShell, from the repo root)
```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip wheel setuptools

# PyTorch 2.5.1 + CUDA 12.4 (Windows wheels)
pip install torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 `
    --index-url https://download.pytorch.org/whl/cu124

# PyG + scatter/sparse Windows wheels matching torch 2.5.1+cu124
pip install torch-geometric
pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu124.html
pip install torch-sparse  -f https://data.pyg.org/whl/torch-2.5.1+cu124.html

# Scientific + optimisation + plotting stack
pip install numpy "scipy>=1.10" "pandas>=2.0" pyarrow `
    "pyomo>=6.7" "highspy>=1.5" `
    "matplotlib>=3.7" "seaborn>=0.13" `
    PyYAML tqdm jupyter ipykernel
```

### Required files on disk (under the repo root)
- `src/` — full source (already in git)
- `outputs/scenarios_v3/reports/` — scenario reports for silver scenarios
- `outputs/encoders/hierarchical_temporal_v3/embeddings_v3/` — HTE embeddings
- `outputs/ebm_models/ebm_v3/silver_best.pt` — stage-1 silver checkpoint **(must exist before running v2)**
- `outputs/scenarios_v3/classification_index.json` — gold/silver classification index

### RTX 4050 (6 GB VRAM) tips
- The EBM model (0.5 M params) is small, but silver scenarios can have varying zone counts.
- If you hit CUDA OOM, reduce `silver_lp_scenarios_per_batch` from 2 to 1 and/or `silver_lp_candidates_per_scenario` from 4 to 2.

## 1. Verify Local Environment

In [1]:
# Quick sanity check that all required packages import and CUDA is visible.
# Install them in your venv first - see prerequisites at the top of the notebook.
import sys, importlib

required = [
    'numpy', 'scipy', 'pandas', 'pyarrow', 'tqdm', 'yaml',
    'matplotlib', 'seaborn',
    'torch', 'torch_geometric',
    'pyomo', 'highspy',
]
missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print(f'MISSING packages: {missing}')
    print('Install them in your venv before continuing (see prerequisites).')
else:
    print('All required packages import OK.')

import torch
print(f'\nPython:   {sys.version.split()[0]}')
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA OK:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'CUDA ver: {torch.version.cuda}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

All required packages import OK.

Python:   3.11.9
PyTorch:  2.5.1+cu124
CUDA OK:  True
GPU:      NVIDIA GeForce RTX 4050 Laptop GPU
CUDA ver: 12.4
VRAM:     6.4 GB


## 2. Setup Local Paths

In [2]:
# Clear module cache so updated .py files are reloaded
import importlib, sys

modules_to_clear = [m for m in sys.modules if m.startswith('src.')]
for m in modules_to_clear:
    del sys.modules[m]
print(f"Cleared {len(modules_to_clear)} cached modules")

Cleared 0 cached modules


In [3]:
import os, sys, json, time
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from tqdm.auto import tqdm

# --- Local repo path ---
# Auto-detect: the notebook lives at <repo>/notebooks/, so the repo is the parent.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    REPO_PATH = NOTEBOOK_DIR.parent
else:
    # Fallback: hard-coded path (edit if your checkout lives elsewhere)
    REPO_PATH = Path(r'C:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn')

assert REPO_PATH.exists(), f'Repo path not found: {REPO_PATH}'
sys.path.insert(0, str(REPO_PATH))
os.chdir(REPO_PATH)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Repo:   {REPO_PATH}')
print(f'Device: {DEVICE}')
print(f'GPU:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Repo:   c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn
Device: cuda
GPU:    NVIDIA GeForce RTX 4050 Laptop GPU


## 3. Configuration

In [4]:
from src.ebm.config_v3 import EBMv3Config

config = EBMv3Config(
    base_dir=str(REPO_PATH),
    # Data dimensions
    n_timesteps=24,
    n_features=7,
    embed_dim=128,
    # Model
    hidden_dim=128,
    gru_layers=2,
    bidirectional=True,
    dropout=0.1,
    use_peak_term=True,
    peak_tau=0.5,
    peak_weight=0.3,
    energy_max=50.0,
    # Sampler
    langevin_steps=100,
    langevin_train_steps_start=10,
    langevin_train_steps_end=50,
    langevin_step_size=0.05,
    langevin_noise=0.50,
    langevin_temp_max=1.0,
    langevin_temp_min=0.1,
    langevin_init_mode='soft',
    langevin_prior_p=0.025,
    langevin_prior_strength=0.0,
    langevin_normalize_grad=True,
    # Negative sampling
    langevin_ratio_start=1.0,
    langevin_ratio_end=1.0,
    random_neg_sparsity=0.025,
    corruption_flip_rate=0.05,
    # Training
    batch_size=32,
    learning_rate=2e-5,
    use_amp=True,
    seed=42,
    # Step A (not used in v2, but kept for compatibility)
    gold_epochs=50,
    gold_lr=2e-5,
    gold_patience=10,
    # Step B base config
    silver_epochs=30,
    silver_lr=1e-5,
    silver_patience=8,
    silver_min_delta=0.01,
    silver_langevin_start=20,
    silver_langevin_end=35,
    silver_lp_eval_every=5,
    silver_lp_scenarios_per_batch=4,
    silver_preference_margin=0.1,
    silver_lambda_cd=1.0,
    silver_lambda_pref=0.5,
)

print(f'Reports dir:    {config.reports_dir}')
print(f'Embeddings dir: {config.embeddings_dir}')
print(f'Output dir:     {config.output_dir}')
print(f'Device:         {config.device}')
print(f'Model: energy_max={config.energy_max}')

Reports dir:    c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\scenarios_v3\reports
Embeddings dir: c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\encoders\hierarchical_temporal_v3\embeddings_v3
Output dir:     c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\ebm_models\ebm_v3
Device:         cuda
Model: energy_max=50.0


## 4. Verify Required Files

In [5]:
# Verify silver checkpoint exists (required for v2 fine-tuning)
silver_ckpt_path = REPO_PATH / 'outputs/ebm_models/ebm_v3/silver_best.pt'
if silver_ckpt_path.exists():
    print(f'✓ Silver checkpoint found: {silver_ckpt_path}')
    print(f'  Size: {silver_ckpt_path.stat().st_size / 1e6:.2f} MB')
else:
    print(f'✗ Silver checkpoint NOT found: {silver_ckpt_path}')
    print('  Please run stage-1 silver fine-tuning first (or download from Google Drive)')
    print('  The silver_v2 fine-tuning requires a pre-trained silver checkpoint.')

# Verify classification index
index_path = REPO_PATH / 'outputs/scenarios_v3/classification_index.json'
if index_path.exists():
    with open(index_path, 'r') as f:
        index = json.load(f)
    print(f'✓ Classification index found')
    print(f'  Gold scenarios: {len(index["gold"])}')
    print(f'  Silver scenarios: {len(index["silver"])}')
else:
    print(f'✗ Classification index NOT found: {index_path}')

# Verify embeddings directory
emb_dir = REPO_PATH / 'outputs/encoders/hierarchical_temporal_v3/embeddings_v3'
if emb_dir.exists():
    emb_files = list(emb_dir.glob('*.npz'))
    print(f'✓ Embeddings directory found: {len(emb_files)} files')
else:
    print(f'✗ Embeddings directory NOT found: {emb_dir}')

# Verify reports directory
reports_dir = REPO_PATH / 'outputs/scenarios_v3/reports'
if reports_dir.exists():
    report_files = list(reports_dir.glob('*.json'))
    print(f'✓ Reports directory found: {len(report_files)} files')
else:
    print(f'✗ Reports directory NOT found: {reports_dir}')

✓ Silver checkpoint found: c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\ebm_models\ebm_v3\silver_best.pt
  Size: 6.68 MB
✓ Classification index found
  Gold scenarios: 4388
  Silver scenarios: 612
✓ Embeddings directory found: 5000 files
✓ Reports directory found: 5001 files


## 5. Load Silver Checkpoint

In [6]:
from src.ebm.model_v3 import TrajectoryZonalEBM

assert silver_ckpt_path.exists(), f'Silver checkpoint not found: {silver_ckpt_path}'

# Initialize model
model = TrajectoryZonalEBM(
    embed_dim=config.embed_dim,
    n_features=config.n_features,
    hidden_dim=config.hidden_dim,
    gru_layers=config.gru_layers,
    bidirectional=config.bidirectional,
    dropout=config.dropout,
    use_peak_term=config.use_peak_term,
    peak_tau=config.peak_tau,
    peak_weight=config.peak_weight,
    energy_max=config.energy_max,
).to(config.device)

# Load checkpoint
ckpt = torch.load(silver_ckpt_path, map_location=config.device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params:,} parameters')
print(f'Checkpoint epoch: {ckpt.get("epoch", "unknown")}')
print(f'Checkpoint val gap: {ckpt.get("val_gap", "unknown")}')

Model loaded: 553,729 parameters
Checkpoint epoch: 6
Checkpoint val gap: 8.296330213546753


## 6. Configure Silver Fine-Tuning v2

In [ ]:
from copy import deepcopy

# Create silver_v2 config with ranking-driven parameters
silver_v2_config = deepcopy(config)
silver_v2_config.silver_output_prefix = 'silver_v2'
silver_v2_config.silver_epochs = 10
silver_v2_config.silver_lr = 5e-6
silver_v2_config.silver_patience = 5
silver_v2_config.silver_min_delta = 0.005
silver_v2_config.silver_lambda_cd = 0.1
silver_v2_config.silver_lambda_pref = 2.0
silver_v2_config.silver_pref_warmup_epochs = 0
silver_v2_config.langevin_prior_strength = 0.0
silver_v2_config.silver_lp_eval_every = 5
silver_v2_config.silver_lp_scenarios_per_batch = 2  # Reduced for RTX 4050 (6GB)
silver_v2_config.silver_lp_candidates_per_scenario = 8
silver_v2_config.silver_lp_max_stages = 4
silver_v2_config.silver_lp_incumbent_candidates = 1
silver_v2_config.silver_lp_oracle_langevin_candidates = 3
silver_v2_config.silver_lp_corrupt_candidates = 2
silver_v2_config.silver_lp_langevin_candidates = 2
silver_v2_config.silver_pref_all_informative_pairs = True
silver_v2_config.silver_pref_max_pairs_per_scenario = 4
silver_v2_config.silver_pref_min_relative_gap = 0.10
silver_v2_config.silver_pref_slack_weight = 1000
silver_v2_config.silver_pref_margin_mode = 'scaled_gap'
silver_v2_config.silver_preference_margin = 0.5
silver_v2_config.silver_pref_margin_rel_gap_cap = 2.0
silver_v2_config.silver_val_ranking_scenarios = 12
silver_v2_config.silver_val_candidates_per_scenario = 8
silver_v2_config.silver_val_max_stages = 4
silver_v2_config.silver_early_stop_metric = 'val_pref_accuracy'

# Optional last-mile pure ranking pass:
silver_v2_config.silver_lambda_cd = 0.05
silver_v2_config.silver_lambda_pref = 3.0

print('Silver v2 Configuration:')
print(f'  Output prefix: {silver_v2_config.silver_output_prefix}')
print(f'  Epochs: {silver_v2_config.silver_epochs}')
print(f'  Learning rate: {silver_v2_config.silver_lr}')
print(f'  Lambda CD: {silver_v2_config.silver_lambda_cd}')
print(f'  Lambda Pref: {silver_v2_config.silver_lambda_pref}')
print(f'  Min relative gap: {silver_v2_config.silver_pref_min_relative_gap:.0%}')
print(f'  Slack weight: {silver_v2_config.silver_pref_slack_weight}')
print(f'  Early stop metric: {silver_v2_config.silver_early_stop_metric}')
print(f'  LP scenarios per batch: {silver_v2_config.silver_lp_scenarios_per_batch}')
print(f'  LP candidates per scenario: {silver_v2_config.silver_lp_candidates_per_scenario}')
print(f'  LP max stages: {silver_v2_config.silver_lp_max_stages}')

Silver v2 Configuration:
  Output prefix: silver_v2
  Epochs: 6
  Learning rate: 5e-06
  Lambda CD: 0.05
  Lambda Pref: 3.0
  Min relative gap: 10%
  Slack weight: 1000
  Early stop metric: val_pref_accuracy
  LP scenarios per batch: 2
  LP candidates per scenario: 8
  LP max stages: 4


## 7. Run Silver Fine-Tuning v2

In [8]:
from src.ebm.train_v3 import run_silver_finetuning

print('='*80)
print('STEP B: SILVER FINE-TUNING v2')
print('='*80)
print(f'Starting from checkpoint: {silver_ckpt_path}')
print(f'Output prefix: {silver_v2_config.silver_output_prefix}')

model, silver_v2_history = run_silver_finetuning(model, silver_v2_config)

print('\n' + '='*80)
print('Silver v2 fine-tuning complete!')
print('='*80)

STEP B: SILVER FINE-TUNING v2
Starting from checkpoint: c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\ebm_models\ebm_v3\silver_best.pt
Output prefix: silver_v2
STEP B: SILVER FINE-TUNING
Building silver dataset: 612 scenarios in index
  Found 5000 embedding files (sample: c:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\encoders\hierarchical_temporal_v3\embeddings_v3\outputs_scenarios_v3_scenario_00001.npz)
  Valid scenarios with reports + embeddings: 612
  Train: 551 samples, 17 batches
  Val:   61 samples, 2 batches
Silver sampler: 20→35 Langevin steps (curriculum), step_size=0.05, noise=0.5
LP worker cached for silver fine-tuning
  [Epoch 0] Batch 10/17 | Total=-0.1632 | CD=-3.2644 | Gap=3.2644 [L]
Epoch 1/6 (168.2s) | Total=-0.2054 CD=-4.1089 Pref=n/a | Pairs=0 Attempted=6 WithPairs=0 PairCoverage=0% NonFinite=100% | TrainPrefAcc=n/a ValPrefAcc=n/a Spearman(E,J)=n/a BestOfKGap=0.0% | ValGap_R=9.3108 ValGap_L=4.7082 | LR=4.67e-06 | Lsteps

KeyboardInterrupt: 

## 8. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import os

# Plot silver training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
train_total = [m.get('loss_total', 0) for m in silver_v2_history['train']]
axes[0].plot(train_total, label='Train Total')
axes[0].set_title('Total Loss (CD + Pref)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# CD component
train_cd = [m.get('cd/cd_loss', 0) for m in silver_v2_history['train']]
val_cd = [m.get('cd_loss', 0) for m in silver_v2_history['val']]
axes[1].plot(train_cd, label='Train CD')
axes[1].plot(val_cd, label='Val CD')
axes[1].set_title('CD Loss Component')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Ranking metrics
train_pref_acc = [m.get('pref/pref_accuracy', 0) for m in silver_v2_history['train']]
val_pref_acc = [m.get('val_pref_accuracy', float('nan')) for m in silver_v2_history['val']]
val_spearman = [m.get('val_spearman', float('nan')) for m in silver_v2_history['val']]
if any(v > 0 for v in train_pref_acc) or any(v == v for v in val_pref_acc):
    axes[2].plot(train_pref_acc, label='Train Pref Acc')
    if any(v == v for v in val_pref_acc):
        axes[2].plot(val_pref_acc, label='Val Pref Acc')
    if any(v == v for v in val_spearman):
        axes[2].plot(val_spearman, label='Val Spearman')
    axes[2].set_title('Ranking Metrics')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylim(-1, 1)
    axes[2].legend()
else:
    val_gap = [m.get('E_gap', 0) for m in silver_v2_history['val']]
    axes[2].plot(val_gap, label='Val Gap')
    axes[2].set_title('Val Energy Gap')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()

plot_prefix = silver_v2_config.silver_output_prefix
plot_path = REPO_PATH / config.output_dir / f'{plot_prefix}_training_curves.png'
plt.tight_layout()
plt.savefig(plot_path, dpi=150)
print(f'Training curves saved to: {plot_path}')
plt.show()

## 9. Save Training History

In [ ]:
import json

# Save training history
history_path = REPO_PATH / config.output_dir / f'{silver_v2_config.silver_output_prefix}_history.json'
with open(history_path, 'w') as f:
    json.dump(silver_v2_history, f, indent=2, default=str)
print(f'Training history saved to: {history_path}')

# Print summary
print('\nTraining Summary:')
print(f'  Epochs trained: {len(silver_v2_history["train"])}')
if silver_v2_history['train']:
    final_train = silver_v2_history['train'][-1]
    print(f'  Final train loss: {final_train.get("loss_total", "N/A")}')
    print(f'  Final train CD loss: {final_train.get("cd/cd_loss", "N/A")}')
    if 'pref/pref_accuracy' in final_train:
        print(f'  Final train pref acc: {final_train["pref/pref_accuracy"]:.2%}')

if silver_v2_history['val']:
    final_val = silver_v2_history['val'][-1]
    print(f'  Final val CD loss: {final_val.get("cd_loss", "N/A")}')
    print(f'  Final val gap: {final_val.get("E_gap", "N/A")}')
    if 'val_pref_accuracy' in final_val:
        print(f'  Final val pref acc: {final_val["val_pref_accuracy"]:.2%}')
    if 'val_spearman' in final_val:
        print(f'  Final val Spearman: {final_val["val_spearman"]:.3f}')

## 10. List Output Files

In [ ]:
output_dir = REPO_PATH / config.output_dir
print(f'Output directory: {output_dir}')
print('\nFiles:')
for f in sorted(output_dir.iterdir()):
    if silver_v2_config.silver_output_prefix in f.name or 'silver' in f.name:
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name}: {size_kb:.1f} KB')